In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [2]:
def preprocessing(data, typ, scaler=None, imputer=None):
    main_features = ['E1', 'E2', 'E3', 'E4', 'E5', 'E7', 'E8', 'E9', 'E16', 'E17', 'E19', 'E20', 
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8', 'S12', "I2", "P8", "P9", "P10",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*']
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    # === Extended Feature Engineering ===
    
    # Original features (keeping your existing code)
    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
    
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
    
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
    
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
    
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
    
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
    
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
    
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
    
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
    
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
    
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
    
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
    
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')
    
    # === NEW FEATURES ===
    
    # 1. Polynomial Features (squared and cubed)
    data[f'M*_squared'] = data['M*'] ** 2
    data[f'E*_squared'] = data['E*'] ** 2
    data[f'I*_squared'] = data['I*'] ** 2
    data[f'P*_squared'] = data['P*'] ** 2
    data[f'V*_squared'] = data['V*'] ** 2
    data[f'S*_squared'] = data['S*'] ** 2
    main_features.extend([f'M*_squared', f'E*_squared', f'I*_squared', 
                          f'P*_squared', f'V*_squared', f'S*_squared'])
    
    # 2. Triple interactions (momentum-volatility-sentiment)
    data[f'MVS_triple_M*_V*_S*'] = data['M*'] * data['V*'] * data['S*']
    data[f'MVS_risk_M*_V*_S*'] = data['M*'] * data['V*'] * abs(data['S*'])
    main_features.append(f'MVS_triple_M*_V*_S*')
    main_features.append(f'MVS_risk_M*_V*_S*')
    
    # 3. Efficiency-inflation-price triple
    data[f'EIP_triple_E*_I*_P*'] = data['E*'] * data['I*'] * data['P*']
    data[f'EIP_real_E*_I*_P*'] = data['E*'] * data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'EIP_triple_E*_I*_P*')
    main_features.append(f'EIP_real_E*_I*_P*')
    
    # 4. Momentum-efficiency-inflation
    data[f'MEI_triple_M*_E*_I*'] = data['M*'] * data['E*'] * data['I*']
    data[f'MEI_adjusted_M*_E*_I*'] = data['M*'] * data['E*'] / (1 + abs(data['I*']) + 1e-8)
    main_features.append(f'MEI_triple_M*_E*_I*')
    main_features.append(f'MEI_adjusted_M*_E*_I*')
    
    # 5. Complex ratios
    data[f'MEIP_ratio'] = (data['M*'] * data['E*']) / (data['I*'] * data['P*'] + 1e-8)
    data[f'MVPS_ratio'] = (data['M*'] * data['V*']) / (data['P*'] * abs(data['S*']) + 1e-8)
    main_features.append(f'MEIP_ratio')
    main_features.append(f'MVPS_ratio')
    
    # 6. Harmonic means
    data[f'ME_harmonic'] = 2 / ((1/(abs(data['M*']) + 1e-8)) + (1/(abs(data['E*']) + 1e-8)))
    data[f'IP_harmonic'] = 2 / ((1/(abs(data['I*']) + 1e-8)) + (1/(abs(data['P*']) + 1e-8)))
    main_features.append(f'ME_harmonic')
    main_features.append(f'IP_harmonic')
    
    # 7. Geometric means
    data[f'ME_geometric'] = np.sqrt(abs(data['M*'] * data['E*'])) * np.sign(data['M*'] * data['E*'])
    data[f'PV_geometric'] = np.sqrt(abs(data['P*'] * data['V*'])) * np.sign(data['P*'] * data['V*'])
    main_features.append(f'ME_geometric')
    main_features.append(f'PV_geometric')
    
    # 8. Exponential transformations
    data[f'M*_exp'] = np.sign(data['M*']) * (np.exp(abs(data['M*'])) - 1)
    data[f'V*_exp'] = np.exp(data['V*']) - 1
    data[f'S*_exp'] = np.sign(data['S*']) * (np.exp(abs(data['S*'])) - 1)
    main_features.append(f'M*_exp')
    main_features.append(f'V*_exp')
    main_features.append(f'S*_exp')
    
    # 9. Logarithmic transformations
    data[f'M*_log'] = np.sign(data['M*']) * np.log1p(abs(data['M*']))
    data[f'V*_log'] = np.log1p(data['V*'])
    data[f'P*_log'] = np.sign(data['P*']) * np.log1p(abs(data['P*']))
    main_features.append(f'M*_log')
    main_features.append(f'V*_log')
    main_features.append(f'P*_log')
    
    # 10. Momentum strength indicators
    data[f'M*_strength'] = abs(data['M*']) / (data['V*'] + 1e-8)
    data[f'M*_conviction'] = data['M*'] * abs(data['S*']) / (data['V*'] + 1e-8)
    main_features.append(f'M*_strength')
    main_features.append(f'M*_conviction')
    
    # 11. Risk-adjusted returns
    data[f'ME_sharpe'] = (data['M*'] - data['I*']) / (data['V*'] + 1e-8)
    data[f'EP_risk_adj'] = data['E*'] * data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'ME_sharpe')
    main_features.append(f'EP_risk_adj')
    
    # 12. Sentiment-adjusted features
    data[f'M*_sent_adj'] = data['M*'] * (1 + 0.5 * data['S*'])
    data[f'P*_sent_adj'] = data['P*'] * (1 - 0.3 * abs(data['S*']))
    data[f'E*_sent_adj'] = data['E*'] * (1 + 0.2 * data['S*'])
    main_features.append(f'M*_sent_adj')
    main_features.append(f'P*_sent_adj')
    main_features.append(f'E*_sent_adj')
    
    # 13. Volatility-normalized features
    data[f'M*_vol_norm'] = data['M*'] / np.sqrt(data['V*'] + 1e-8)
    data[f'E*_vol_norm'] = data['E*'] / np.sqrt(data['V*'] + 1e-8)
    data[f'P*_vol_norm'] = data['P*'] / np.sqrt(data['V*'] + 1e-8)
    main_features.append(f'M*_vol_norm')
    main_features.append(f'E*_vol_norm')
    main_features.append(f'P*_vol_norm')
    
    # 14. Composite momentum indicators
    data[f'momentum_composite'] = (data['M*'] * 0.4 + data['E*'] * 0.3 + 
                                   data['P*'] * 0.2 + data['S*'] * 0.1)
    data[f'momentum_quality'] = data['M*'] * data['E*'] / (data['V*'] * abs(data['I*']) + 1e-8)
    main_features.append(f'momentum_composite')
    main_features.append(f'momentum_quality')
    
    # 15. Divergence indicators
    data[f'ME_divergence'] = abs(data['M*'] - data['E*']) / (abs(data['M*']) + abs(data['E*']) + 1e-8)
    data[f'PS_divergence'] = abs(data['P*'] - data['S*']) / (abs(data['P*']) + abs(data['S*']) + 1e-8)
    main_features.append(f'ME_divergence')
    main_features.append(f'PS_divergence')
    
    # 16. Min/Max features
    data[f'MEI_max'] = np.maximum(np.maximum(data['M*'], data['E*']), data['I*'])
    data[f'MEI_min'] = np.minimum(np.minimum(data['M*'], data['E*']), data['I*'])
    data[f'MEI_range'] = data[f'MEI_max'] - data[f'MEI_min']
    main_features.append(f'MEI_max')
    main_features.append(f'MEI_min')
    main_features.append(f'MEI_range')
    
    # 17. Percentile features
    data[f'M*_percentile'] = data['M*'].rank(pct=True)
    data[f'V*_percentile'] = data['V*'].rank(pct=True)
    data[f'S*_percentile'] = data['S*'].rank(pct=True)
    main_features.append(f'M*_percentile')
    main_features.append(f'V*_percentile')
    main_features.append(f'S*_percentile')
    
    # 18. Interaction with squared terms
    data[f'M*E*_squared'] = (data['M*'] * data['E*']) ** 2
    data[f'PV_squared'] = (data['P*'] * data['V*']) ** 2
    main_features.append(f'M*E*_squared')
    main_features.append(f'PV_squared')
    
    # 19. Weighted combinations
    data[f'weighted_signal'] = (data['M*'] * 0.35 + data['E*'] * 0.25 + 
                                data['P*'] * 0.20 + data['S*'] * 0.15 - data['I*'] * 0.05)
    data[f'risk_weighted'] = (data['V*'] * 0.6 + abs(data['S*']) * 0.3 + abs(data['I*']) * 0.1)
    main_features.append(f'weighted_signal')
    main_features.append(f'risk_weighted')
    
    # 20. Additional missing pairwise interactions
    data[f'IV_ratio_I*_V*'] = data['I*'] / (data['V*'] + 1e-8)
    data[f'IS_ratio_I*_S*'] = data['I*'] / (abs(data['S*']) + 1e-8)
    data[f'ME_sum_M*_E*'] = data['M*'] + data['E*']
    data[f'EI_sum_E*_I*'] = data['E*'] + data['I*']
    main_features.append(f'IV_ratio_I*_V*')
    main_features.append(f'IS_ratio_I*_S*')
    main_features.append(f'ME_sum_M*_E*')
    main_features.append(f'EI_sum_E*_I*')
    
    print(f"Total features created: {len(main_features)}")

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    # ============================================================================
    # ADVANCED FEATURES BASED ON MULTI-COMPONENT COMBINATIONS
    # ============================================================================
    
    # --- Features based on M*_P*_V* (Market-Price-Volatility) ---
    data['M*_P*_V*_prod'] = data['M*'] * data['P*'] * data['V*']
    data['M*_P*_V*_mean'] = data['M*_P*_V*'] / 3
    data['M*_P*_V*_weighted'] = data['M*'] * 0.5 + data['P*'] * 0.3 + data['V*'] * 0.2
    data['M*_P*_V*_volatility'] = data['M*_P*_V*'] * data['V*']
    data['M*_P*_V*_risk'] = data['M*_P*_V*'] / (data['V*'] + 1e-8)
    
    main_features.extend(['M*_P*_V*_prod', 'M*_P*_V*_mean', 'M*_P*_V*_weighted', 
                          'M*_P*_V*_volatility', 'M*_P*_V*_risk'])
    
    # --- Features based on M*_E*_S* (Market-Economic-Sentiment) ---
    data['M*_E*_S*_prod'] = data['M*'] * data['E*'] * data['S*']
    data['M*_E*_S*_mean'] = data['M*_E*_S*'] / 3
    data['M*_E*_S*_momentum'] = data['M*_E*_S*'] * data['S*']
    data['M*_E*_S*_stability'] = data['M*_E*_S*'] / (abs(data['S*']) + 1e-8)
    data['M*_E*_S*_weighted'] = data['M*'] * 0.4 + data['E*'] * 0.4 + data['S*'] * 0.2
    
    main_features.extend(['M*_E*_S*_prod', 'M*_E*_S*_mean', 'M*_E*_S*_momentum',
                          'M*_E*_S*_stability', 'M*_E*_S*_weighted'])
    
    # --- Features based on M*_P*_I*_V* (Market-Price-Info-Volatility) ---
    data['M*_P*_I*_V*_prod'] = data['M*'] * data['P*'] * data['I*'] * data['V*']
    data['M*_P*_I*_V*_mean'] = data['M*_P*_I*_V*'] / 4
    data['M*_P*_I*_V*_uncertainty'] = data['M*_P*_I*_V*'] * data['V*'] * data['I*']
    data['M*_P*_I*_V*_signal'] = (data['M*'] + data['P*']) / (data['I*'] + data['V*'] + 1e-8)
    data['M*_P*_I*_V*_balanced'] = data['M*'] * 0.3 + data['P*'] * 0.3 + data['I*'] * 0.2 + data['V*'] * 0.2
    
    main_features.extend(['M*_P*_I*_V*_prod', 'M*_P*_I*_V*_mean', 'M*_P*_I*_V*_uncertainty',
                          'M*_P*_I*_V*_signal', 'M*_P*_I*_V*_balanced'])
    
    # --- Features based on V*_M*_E*_I* (Volatility-Market-Economic-Info) ---
    data['V*_M*_E*_I*_prod'] = data['V*'] * data['M*'] * data['E*'] * data['I*']
    data['V*_M*_E*_I*_mean'] = data['V*_M*_E*_I*'] / 4
    data['V*_M*_E*_I*_risk_adj'] = (data['M*'] + data['E*'] + data['I*']) / (data['V*'] + 1e-8)
    data['V*_M*_E*_I*_vol_weighted'] = data['V*_M*_E*_I*'] * data['V*']
    data['V*_M*_E*_I*_fundamental'] = (data['M*'] * data['E*']) + (data['I*'] * data['V*'])
    
    main_features.extend(['V*_M*_E*_I*_prod', 'V*_M*_E*_I*_mean', 'V*_M*_E*_I*_risk_adj',
                          'V*_M*_E*_I*_vol_weighted', 'V*_M*_E*_I*_fundamental'])
    
    # --- Features based on V*_S*_D* (Volatility-Sentiment-Direction) ---
    data['V*_S*_D*_prod'] = data['V*'] * data['S*'] * data['D*']
    data['V*_S*_D*_mean'] = data['V*_S*_D*'] / 3
    data['V*_S*_D*_panic'] = data['V*'] * abs(data['S*']) * abs(data['D*'])
    data['V*_S*_D*_trend'] = data['S*'] * data['D*'] / (data['V*'] + 1e-8)
    data['V*_S*_D*_regime'] = data['V*_S*_D*'] * (1 + abs(data['S*']))
    
    main_features.extend(['V*_S*_D*_prod', 'V*_S*_D*_mean', 'V*_S*_D*_panic',
                          'V*_S*_D*_trend', 'V*_S*_D*_regime'])
    
    # --- Features based on E*_I*_V*_D* (Economic-Info-Volatility-Direction) ---
    data['E*_I*_V*_D*_prod'] = data['E*'] * data['I*'] * data['V*'] * data['D*']
    data['E*_I*_V*_D*_mean'] = data['E*_I*_V*_D*'] / 4
    data['E*_I*_V*_D*_macro'] = (data['E*'] + data['D*']) * (data['I*'] + data['V*'])
    data['E*_I*_V*_D*_directional'] = (data['E*'] * data['D*']) / (data['V*'] + 1e-8)
    data['E*_I*_V*_D*_uncertainty'] = data['I*'] * data['V*'] * abs(data['D*'])
    
    main_features.extend(['E*_I*_V*_D*_prod', 'E*_I*_V*_D*_mean', 'E*_I*_V*_D*_macro',
                          'E*_I*_V*_D*_directional', 'E*_I*_V*_D*_uncertainty'])
    
    # --- Features based on M*_V*_S*_D* (Market-Volatility-Sentiment-Direction) ---
    data['M*_V*_S*_D*_prod'] = data['M*'] * data['V*'] * data['S*'] * data['D*']
    data['M*_V*_S*_D*_mean'] = data['M*_V*_S*_D*'] / 4
    data['M*_V*_S*_D*_sentiment_adj'] = data['M*'] * data['S*'] / (data['V*'] + 1e-8)
    data['M*_V*_S*_D*_momentum'] = (data['M*'] + data['S*'] + data['D*']) * data['V*']
    data['M*_V*_S*_D*_reversal'] = data['M*'] * (-data['S*']) * data['D*']
    
    main_features.extend(['M*_V*_S*_D*_prod', 'M*_V*_S*_D*_mean', 'M*_V*_S*_D*_sentiment_adj',
                          'M*_V*_S*_D*_momentum', 'M*_V*_S*_D*_reversal'])
    
    # --- Features based on P*_V*_S* (Price-Volatility-Sentiment) ---
    data['P*_V*_S*_prod'] = data['P*'] * data['V*'] * data['S*']
    data['P*_V*_S*_mean'] = data['P*_V*_S*'] / 3
    data['P*_V*_S*_price_risk'] = data['P*'] / (data['V*'] + 1e-8)
    data['P*_V*_S*_sentiment_vol'] = data['S*'] * data['V*']
    data['P*_V*_S*_contrarian'] = data['P*'] * (-data['S*']) * data['V*']
    
    main_features.extend(['P*_V*_S*_prod', 'P*_V*_S*_mean', 'P*_V*_S*_price_risk',
                          'P*_V*_S*_sentiment_vol', 'P*_V*_S*_contrarian'])
    
    # --- Features based on P*_M*_D* (Price-Market-Direction) ---
    data['P*_M*_D*_prod'] = data['P*'] * data['M*'] * data['D*']
    data['P*_M*_D*_mean'] = data['P*_M*_D*'] / 3
    data['P*_M*_D*_trend'] = (data['P*'] + data['M*']) * data['D*']
    data['P*_M*_D*_strength'] = abs(data['P*_M*_D*'])
    data['P*_M*_D*_momentum'] = data['P*'] * data['M*'] * abs(data['D*'])
    
    main_features.extend(['P*_M*_D*_prod', 'P*_M*_D*_mean', 'P*_M*_D*_trend',
                          'P*_M*_D*_strength', 'P*_M*_D*_momentum'])
    
    # --- Features based on M*_E*_P*_S* (Market-Economic-Price-Sentiment) ---
    data['M*_E*_P*_S*_prod'] = data['M*'] * data['E*'] * data['P*'] * data['S*']
    data['M*_E*_P*_S*_mean'] = data['M*_E*_P*_S*'] / 4
    data['M*_E*_P*_S*_core'] = (data['M*'] + data['E*']) * (data['P*'] + data['S*'])
    data['M*_E*_P*_S*_fundamental'] = (data['M*'] * data['E*']) / (abs(data['S*']) + 1e-8)
    data['M*_E*_P*_S*_weighted'] = data['M*'] * 0.3 + data['E*'] * 0.3 + data['P*'] * 0.2 + data['S*'] * 0.2
    
    main_features.extend(['M*_E*_P*_S*_prod', 'M*_E*_P*_S*_mean', 'M*_E*_P*_S*_core',
                          'M*_E*_P*_S*_fundamental', 'M*_E*_P*_S*_weighted'])
    
    # --- Features based on M*_E*_I*_P*_V*_S*_D* (All components) ---
    data['M*_E*_I*_P*_V*_S*_D*_prod_all'] = data['M*'] * data['E*'] * data['I*'] * data['P*'] * data['V*'] * data['S*'] * data['D*']
    data['M*_E*_I*_P*_V*_S*_D*_mean'] = data['M*_E*_I*_P*_V*_S*_D*'] / 7
    data['M*_E*_I*_P*_V*_S*_D*_core_ratio'] = (data['M*'] + data['E*'] + data['I*']) / (data['V*'] + abs(data['S*']) + abs(data['D*']) + 1e-8)
    data['M*_E*_I*_P*_V*_S*_D*_vol_adj'] = data['M*_E*_I*_P*_V*_S*_D*'] / (data['V*'] + 1e-8)
    data['M*_E*_I*_P*_V*_S*_D*_sentiment_adj'] = data['M*_E*_I*_P*_V*_S*_D*'] * (1 + data['S*'])
    data['M*_E*_I*_P*_V*_S*_D*_weighted'] = (data['M*'] * 0.2 + data['E*'] * 0.2 + data['I*'] * 0.15 + 
                                              data['P*'] * 0.15 + data['V*'] * 0.1 + data['S*'] * 0.1 + data['D*'] * 0.1)
    data['M*_E*_I*_P*_V*_S*_D*_fundamental'] = (data['M*'] + data['E*'] + data['I*']) / 3
    data['M*_E*_I*_P*_V*_S*_D*_technical'] = (data['P*'] + data['V*'] + data['S*'] + data['D*']) / 4
    
    main_features.extend(['M*_E*_I*_P*_V*_S*_D*_prod_all', 'M*_E*_I*_P*_V*_S*_D*_mean', 
                          'M*_E*_I*_P*_V*_S*_D*_core_ratio', 'M*_E*_I*_P*_V*_S*_D*_vol_adj',
                          'M*_E*_I*_P*_V*_S*_D*_sentiment_adj', 'M*_E*_I*_P*_V*_S*_D*_weighted',
                          'M*_E*_I*_P*_V*_S*_D*_fundamental', 'M*_E*_I*_P*_V*_S*_D*_technical'])
    
    # --- Cross-combination features ---
    data['MPV_ratio_MES'] = data['M*_P*_V*'] / (data['M*_E*_S*'] + 1e-8)
    data['MPIV_ratio_VMEI'] = data['M*_P*_I*_V*'] / (data['V*_M*_E*_I*'] + 1e-8)
    data['VSD_times_EIVD'] = data['V*_S*_D*'] * data['E*_I*_V*_D*']
    data['MVSD_ratio_PVS'] = data['M*_V*_S*_D*'] / (data['P*_V*_S*'] + 1e-8)
    data['PMD_times_MEPS'] = data['P*_M*_D*'] * data['M*_E*_P*_S*']
    
    main_features.extend(['MPV_ratio_MES', 'MPIV_ratio_VMEI', 'VSD_times_EIVD',
                          'MVSD_ratio_PVS', 'PMD_times_MEPS'])
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        data[feature_cols] = imputer.transform(data[feature_cols])
    
    if typ == "train":
        scaler = MinMaxScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
        data[feature_cols] = scaler.transform(data[feature_cols])
    
    if typ == "train":
        return data, scaler, imputer
    else:
        return data


train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed, scaler, imputer = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

Total features created: 124


In [3]:
train_processed

,E1,E2,E3,E4,E5,E7,E8,E9,E16,E17,...,M*_E*_I*_P*_V*_S*_D*_sentiment_adj,M*_E*_I*_P*_V*_S*_D*_weighted,M*_E*_I*_P*_V*_S*_D*_fundamental,M*_E*_I*_P*_V*_S*_D*_technical,MPV_ratio_MES,MPIV_ratio_VMEI,VSD_times_EIVD,MVSD_ratio_PVS,PMD_times_MEPS,target
6969,0.041781,0.727190,0.649542,0.021277,0.979815,0.894408,0.010199,0.018862,0.780305,0.733879,...,0.481792,0.439060,0.447187,0.134611,0.479785,0.885324,0.047226,0.096832,0.131977,0.001145
6970,0.041500,0.728277,0.650503,0.019640,0.980146,0.894352,0.010233,0.018531,0.780123,0.733868,...,0.504652,0.488393,0.454255,0.215396,0.479908,0.886887,0.059469,0.106650,0.205836,0.004738
6971,0.041219,0.736669,0.657605,0.018003,0.980477,0.894295,0.010266,0.018200,0.779941,0.733856,...,0.525577,0.495592,0.440249,0.260434,0.479935,0.887392,0.070438,0.104163,0.218151,0.006016
6972,0.040938,0.747184,0.666562,0.016367,0.980807,0.894394,0.011356,0.017869,0.776877,0.893087,...,0.534423,0.481268,0.429617,0.258151,0.479719,0.885807,0.078676,0.104151,0.188425,0.001414
6973,0.040657,0.750377,0.669411,0.014730,0.981138,0.888036,0.011387,0.017538,0.776701,0.892525,...,0.520973,0.474342,0.435959,0.227989,0.479733,0.885732,0.068633,0.103529,0.180520,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.282092,0.654241,0.781596,0.152209,0.331238,0.851819,0.042188,0.914626,0.627247,0.437709,...,0.503362,0.438193,0.414130,0.170146,0.479740,0.885949,0.044175,0.108365,0.188093,0.002457
8986,0.281430,0.656796,0.783496,0.150573,0.330907,0.851816,0.042187,0.914957,0.627242,0.437767,...,0.560395,0.463485,0.421809,0.209496,0.479696,0.886702,0.049090,0.120077,0.228655,0.002312
8987,0.280770,0.660234,0.786076,0.148936,0.330576,0.851813,0.042186,0.915288,0.627200,0.437777,...,0.534023,0.439990,0.415569,0.179463,0.479622,0.885461,0.049047,0.132765,0.200081,0.002891
8988,0.280112,0.664119,0.788992,0.147300,0.330245,0.851793,0.042186,0.915619,0.627159,0.437787,...,0.480294,0.433472,0.408608,0.161855,0.479855,0.887030,0.037742,0.097515,0.201782,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {'boosting_type': 'gbdt', 
               'objective': 'regression_l1', 
               'metric': 'mape', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 6906349.2710
Training Model 2...
Cumulative Training MAPE: 22490525.6371
Training Model 3...
Cumulative Training MAPE: 1452679.6616
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 1452679.6616
MAE: 0.0016
RMSE: 0.0042


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)


def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
Total features created: 124
